# Práctica 2 — Sesión 6: Exploración de Gymnasium

**Entregable:** `notebooks/p2_06_gymnasium.ipynb`  
**Entorno:** `CartPole-v1`  
**Objetivo:** crear un entorno, realizar **cinco episodios con acciones aleatorias**, imprimir **estado, acción, recompensa y `done`**, y observar la recompensa acumulada.

**Importante:** aquí **no entrenamos** un agente. Primero exploramos cómo se comunica un agente con su entorno. El entrenamiento con PPO corresponde a P2-8.

## 1. Conceptos básicos

- **Agente:** el programa que elige una acción.
- **Entorno:** el sistema que reacciona a la acción; en este caso, un carro con un péndulo invertido.
- **Observación (estado observado):** vector con cuatro valores: posición del carro (m), velocidad del carro (m/s), ángulo del poste (rad) y velocidad angular (rad/s).
- **Acción:** `0` aplica fuerza hacia la izquierda; `1`, hacia la derecha.
- **Recompensa:** en el CartPole estándar, cada paso genera `+1`, hasta que acaba el episodio.
- **Episodio:** secuencia desde `reset()` hasta que el entorno termina (`terminated`) o se alcanza un límite (`truncated`).

La API actual de Gymnasium devuelve `terminated` y `truncated` por separado. Para mostrar el `done` solicitado por la guía, utilizaremos:

```python
done = terminated or truncated
```

**Referencia:** [Documentación oficial de Gymnasium — Basic Usage](https://gymnasium.farama.org/introduction/basic_usage/).

## 2. Importación y creación del entorno

Antes de ejecutar el notebook, instala Gymnasium en el entorno virtual del proyecto:

```fish
python -m pip install gymnasium
```

No utilizaremos `render_mode="human"` en esta sesión, por lo que no se necesita una ventana gráfica.

In [4]:
import gymnasium as gym
import numpy as np

print("Versión de Gymnasium:", gym.__version__)

entorno = gym.make("CartPole-v1")
print("Espacio de observaciones:", entorno.observation_space)
print("Espacio de acciones:", entorno.action_space)
entorno.close()

Versión de Gymnasium: 1.4.0
Espacio de observaciones: Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
Espacio de acciones: Discrete(2)


## 3. Cinco episodios con acciones aleatorias

En cada paso imprimimos la **nueva observación**, la acción elegida, la recompensa y `done`. También imprimimos por separado `terminated` y `truncated`, para distinguir el fracaso del agente de la finalización por límite temporal.

Fijamos semillas para obtener resultados reproducibles. El número de pasos **puede variar** si se cambian las semillas, la versión del entorno o la política.

In [5]:
NUM_EPISODIOS = 5
SEMILLA = 42
recompensas_acumuladas = []
pasos_por_episodio = []

entorno = gym.make("CartPole-v1")
entorno.action_space.seed(SEMILLA)  # Reproducibilidad de las acciones aleatorias.

try:
    for episodio in range(1, NUM_EPISODIOS + 1):
        observacion, info = entorno.reset(seed=SEMILLA + episodio - 1)
        recompensa_total = 0.0
        numero_pasos = 0
        done = False

        print(f"\n{'=' * 70}")
        print(f"EPISODIO {episodio}")
        print("Estado inicial:", np.array2string(observacion, precision=3))

        while not done:
            accion = entorno.action_space.sample()  # No hay entrenamiento.
            observacion, recompensa, terminated, truncated, info = entorno.step(accion)
            done = terminated or truncated
            numero_pasos += 1
            recompensa_total += float(recompensa)

            print(
                f"Paso {numero_pasos:3d} | "
                f"estado={np.array2string(observacion, precision=3)} | "
                f"acción={accion} | recompensa={recompensa:.1f} | "
                f"done={done} (terminated={terminated}, truncated={truncated})"
            )

        recompensas_acumuladas.append(recompensa_total)
        pasos_por_episodio.append(numero_pasos)
        print(f"Recompensa acumulada del episodio {episodio}: {recompensa_total:.1f}")
        print(f"Cantidad de pasos: {numero_pasos}")
finally:
    entorno.close()



EPISODIO 1
Estado inicial: [ 0.027 -0.006  0.036  0.02 ]
Paso   1 | estado=[ 0.027 -0.202  0.036  0.324] | acción=0 | recompensa=1.0 | done=False (terminated=False, truncated=False)
Paso   2 | estado=[ 0.023 -0.007  0.043  0.042] | acción=1 | recompensa=1.0 | done=False (terminated=False, truncated=False)
Paso   3 | estado=[ 0.023  0.187  0.044 -0.236] | acción=1 | recompensa=1.0 | done=False (terminated=False, truncated=False)
Paso   4 | estado=[ 0.027 -0.008  0.039  0.07 ] | acción=0 | recompensa=1.0 | done=False (terminated=False, truncated=False)
Paso   5 | estado=[ 0.027 -0.204  0.04   0.374] | acción=0 | recompensa=1.0 | done=False (terminated=False, truncated=False)
Paso   6 | estado=[ 0.023 -0.01   0.048  0.095] | acción=1 | recompensa=1.0 | done=False (terminated=False, truncated=False)
Paso   7 | estado=[ 0.022 -0.205  0.05   0.402] | acción=0 | recompensa=1.0 | done=False (terminated=False, truncated=False)
Paso   8 | estado=[ 0.018 -0.011  0.058  0.125] | acción=1 | recomp

## 4. Resumen de recompensas

Cada episodio se compara por su recompensa acumulada. En la configuración estándar de CartPole-v1, esta normalmente coincide con la cantidad de pasos ejecutados.

In [6]:
print("\nRESUMEN DE LOS CINCO EPISODIOS")
print(f"{'Episodio':>8} {'Pasos':>8} {'Recompensa total':>20}")
for indice, (pasos, recompensa) in enumerate(zip(pasos_por_episodio, recompensas_acumuladas), start=1):
    print(f"{indice:>8} {pasos:>8} {recompensa:>20.1f}")

print(f"\nRecompensa media:   {np.mean(recompensas_acumuladas):.2f}")
print(f"Recompensa máxima:  {np.max(recompensas_acumuladas):.1f}")
print(f"Recompensa mínima:  {np.min(recompensas_acumuladas):.1f}")



RESUMEN DE LOS CINCO EPISODIOS
Episodio    Pasos     Recompensa total
       1       30                 30.0
       2       46                 46.0
       3       30                 30.0
       4       24                 24.0
       5       28                 28.0

Recompensa media:   31.60
Recompensa máxima:  46.0
Recompensa mínima:  24.0


## 5. Interpretación y conclusión


1. Se ejecutaron cinco episodios de CartPole-v1 utilizando acciones aleatorias. Se registraron las observaciones, acciones, recompensas y condiciones de finalización.

2. La recompensa acumulada varió entre **24 y 46**, con un promedio de **31,60**. El segundo episodio obtuvo el mejor resultado, con 46 pasos.

3. El agente no desarrolló aprendizaje, ya que seleccionó sus acciones aleatoriamente. Estos resultados establecen una línea base para comparar su rendimiento con un agente entrenado mediante PPO en la Sesión 8.
